## Adam Algorithm in Theory

The algorithm to optimize a stochastic objective function $f(\theta)$ with parameters $\theta$ with the Adam method can be summarized verbally as follows:

until the parameters have converged, repeat these steps:

    - compute the gradients w.r.t. the stochastic objective at the last timestep

    - update the mean estimate

    - update the raw variance estimate

    - bias-correct these estimates
    
    - update the parameters

## Adam Algorithm in Practice

### Imports and Setup

In [4]:
import numpy as np
import torch

from typing import Callable

### Class Definition

In [ ]:
class Adam():

    def __init__(self, func: Callable[[torch.Tensor], float], alpha: float = 0.001, beta_1: float = 0.9, beta_2: float = 0.999, epsilon = 10e-8, max_iters: int = 10000) -> None:
        self.alpha = alpha
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.epsilon = epsilon
        self.func = func
        self.max_iters = max_iters
        return
        
    def step(self, theta: np.ndarray) -> np.ndarray:
        m = torch.zeros(theta.shape[0], dtype=torch.float)
        v = torch.zeros(theta.shape[0], dtype=torch.float)
        theta = np.asarray(theta, dtype=np.float32)
        last_theta = torch.from_numpy(theta)
        next_theta = torch.from_numpy(theta) + 2 * self.epsilon
        t = 0

        # until the parameters converge
        while (next_theta - last_theta).abs().max().item() > self.epsilon:
            t += 1
            last_theta = next_theta.detach().requires_grad_(True)
            # zero the gradients
            last_theta.grad = None
            # compute the gradients of f w.r.t. the current parameters
            objective = self.func(last_theta)
            objective.backward()
            g = last_theta.grad
            # compute stochastic moments
            m = self.beta_1 * m + (1 - self.beta_1) * g
            v = self.beta_2 * v + (1 - self.beta_2) * g ** 2
            # update parameters
            a = self.alpha * (torch.sqrt(1 - self.beta_2 ** t) / (1 - self.beta_1 ** t))
            next_theta = last_theta - a * m / (torch.sqrt(v) + self.epsilon)

            if t > self.max_iters:
                break

        return next_theta